# NISAR and ESA BIOMASS: OVERLAPP

Date: Febuary 2,2026

Authors: Harshini Girish(UAH), Rajat Shinde (UAH), Alex Mandel (Development Seed), Samantha Niemoeller (JPL)


Description: This notebook queries NISAR L2 GCOV granules (via earthaccess) and ESA CCI BIOMASS V5.01 tiles (via ESA MAAP STAC) for a chosen AOI and time settings. It converts returned items to footprint polygons and plots them on a single interactive Folium map as two toggleable layers.
An optional overlap layer highlights where NISAR and BIOMASS footprints intersect (bbox or true geometry). The result quickly shows where data coincides spatially to support fusion workflows.

## Run This Notebook

To access and run this tutorial within MAAP's Algorithm Development Environment (ADE), please refer to the ["Getting started with the MAAP"](https://docs.maap-project.org/en/latest/getting_started/getting_started.html) section of our documentation.

Disclaimer: it is highly recommended to run a tutorial within MAAP's ADE, which already includes packages specific to MAAP, such as maap-py. Running the tutorial outside of the MAAP ADE may lead to errors. Additionally, it is recommended to use the `Pangeo` workspace within the ADE, since certain packages relevant to this tutorial are already installed.

## Additional Resources
- [NISAR](https://nisar.jpl.nasa.gov/)
- [BIOMASS](https://docs.maap-project.org/en/develop/science/ESA_CCI/ESA_CCI_V5_Token_Access.html)


## Import and Install Packages

In [1]:
import os
import stat
import getpass
import pathlib

import numpy as np
import matplotlib.pyplot as plt

import earthaccess
from pystac_client import Client

from shapely.geometry import Polygon, box, mapping, shape
from collections import Counter

from folium import Map, GeoJson, LayerControl

plt.rcParams["figure.figsize"] = (6, 6)
plt.rcParams["axes.grid"] = False


## Inputs
This “Inputs” section defines the search settings used later in the notebook: BBOX sets the area of interest as (min_lon, min_lat, max_lon, max_lat) and can be used to spatially filter both datasets, `NISAR_TEMPORAL = ("2025-10-01", "2025-12-31")` restricts the NISAR search to granules acquired within that date range, and `NISAR_COUNT = 6` limits how many NISAR granules (and footprints) will be plotted. For BIOMASS, `BIOMASS_YEAR = 2010` selects a specific annual layer, and BIOMASS_DT converts it into a STAC datetime range (2010-01-01/2010-12-31) used in the BIOMASS query; the tip warns that choosing a year outside the collection’s indexed range (often 2010–2021) will return zero results.



In [2]:
NISAR_TEMPORAL = ("2025-10-01", "2025-12-31")

# How many NISAR granules to plot
NISAR_COUNT = 6

# BIOMASS year (choose explicitly from available years: 2010..2021)
BIOMASS_YEAR = 2010
BIOMASS_DT = f"{BIOMASS_YEAR}-01-01/{BIOMASS_YEAR}-12-31"
BIOMASS_LIMIT = 500 

print("NISAR_TEMPORAL:", NISAR_TEMPORAL)
print("BIOMASS_DT:", BIOMASS_DT)


NISAR_TEMPORAL: ('2025-10-01', '2025-12-31')
BIOMASS_DT: 2010-01-01/2010-12-31


## Access the Data


### 1) NISAR data

This cell sets the NISAR collection short name (`NISAR_L2_GCOV_BETA_V1`), logs in to Earthdata via `earthaccess.login()`, and then queries CMR for matching NISAR granules within the specified `NISAR_TEMPORAL` window, limited to `NISAR_COUNT` results and filtered to cloud-hosted items. Finally, it prints how many granules were returned by the search.


In [3]:
NISAR_SHORT_NAME = "NISAR_L2_GCOV_BETA_V1"

earthaccess.login()

nisar_results = earthaccess.search_data(
    short_name=NISAR_SHORT_NAME,
    cloud_hosted=True,
    temporal=NISAR_TEMPORAL,
    count=NISAR_COUNT,
)

print("NISAR granules found:", len(nisar_results))


NISAR granules found: 6


This cell loops through the NISAR search results and converts each granule into a GeoJSON footprint feature using `nisar_granule_to_feature(g)`, skipping any granules that don’t contain usable footprint metadata. It then bundles all successfully created footprint features into a GeoJSON `FeatureCollection` called `nisar_fc`. Finally, it prints how many NISAR footprint polygons were added to the collection for mapping.


In [11]:
# Build NISAR FeatureCollection for mapping
nisar_features = []
for g in nisar_results:
    try:
        nisar_features.append(nisar_granule_to_feature(g))
    except Exception as e:
        print("Skipping granule (no footprint):", e)

nisar_fc = {"type": "FeatureCollection", "features": nisar_features}
print("NISAR footprints in FeatureCollection:", len(nisar_fc["features"]))


NISAR footprints in FeatureCollection: 6


This cell defines helper functions used before visualization. `_get_umm(g)` safely extracts the UMM metadata dictionary from an `earthaccess` granule. `nisar_granule_to_feature(g)` then converts a single NISAR granule into a GeoJSON Feature by reading its spatial geometry (preferring a polygon from `GPolygons` and falling back to a `BoundingRectangles` box if needed), extracting the granule’s start/end times, and attaching an ID/title in the feature properties. The output Feature objects are later collected into a FeatureCollection and plotted on the interactive map.


In [4]:
def _get_umm(g):
    try:
        return g.get("umm", {})
    except Exception:
        return {}


def nisar_granule_to_feature(g):
    umm = _get_umm(g)

    geom = (
        umm.get("SpatialExtent", {})
           .get("HorizontalSpatialDomain", {})
           .get("Geometry", {})
    )

    poly = None

    # Prefer polygon boundary
    gpolys = geom.get("GPolygons", [])
    if gpolys:
        pts = gpolys[0].get("Boundary", {}).get("Points", [])
        if pts:
            coords = [(p["Longitude"], p["Latitude"]) for p in pts]
            if coords and coords[0] != coords[-1]:
                coords = coords + [coords[0]]
            poly = Polygon(coords)

    # Fallback to bounding rectangle
    if poly is None:
        rects = geom.get("BoundingRectangles", [])
        if rects:
            r = rects[0]
            poly = box(
                r["WestBoundingCoordinate"],
                r["SouthBoundingCoordinate"],
                r["EastBoundingCoordinate"],
                r["NorthBoundingCoordinate"],
            )

    if poly is None:
        raise ValueError("Could not extract footprint geometry from NISAR granule metadata")

    # Time
    time_range = (
        umm.get("TemporalExtent", {})
           .get("RangeDateTime", {})
    )
    t0 = time_range.get("BeginningDateTime")
    t1 = time_range.get("EndingDateTime")

    # Title/ID-like field
    title = umm.get("GranuleUR") or umm.get("Title") or "NISAR granule"

    return {
        "type": "Feature",
        "geometry": mapping(poly),
        "properties": {"title": title, "t0": t0, "t1": t1},
    }


### 2) ESA BIOMASS data
This cell manages your ESA MAAP access token by storing it in a local file (`~/.config/esa_maap/ticket`). If the token file doesn’t exist, it securely prompts you to paste the token, saves it, and sets strict permissions (read/write only for you). It then verifies the file permissions are exactly `600` for security and finally reads the token into `ESA_TOKEN`, confirming where it was loaded from for use in BIOMASS/ESA authenticated requests.


In [27]:
#ESA token file (used if/when you need Authorization for assets) 
TOKEN_FILE = pathlib.Path.home() / ".config" / "esa_maap" / "ticket"
TOKEN_FILE.parent.mkdir(parents=True, exist_ok=True)

if not TOKEN_FILE.exists():
    tok = getpass.getpass("Paste ESA portal token (hidden): ").strip()
    if not tok:
        raise ValueError("Empty token.")
    TOKEN_FILE.write_text(tok, encoding="utf-8")
    TOKEN_FILE.chmod(stat.S_IRUSR | stat.S_IWUSR)

st = TOKEN_FILE.stat()
if (st.st_mode & 0o777) != 0o600:
    raise PermissionError(f"{TOKEN_FILE} must have mode 600. Fix with: chmod 600 {TOKEN_FILE}")

ESA_TOKEN = TOKEN_FILE.read_text(encoding="utf-8").strip()
print("Token loaded from file:", TOKEN_FILE)


Token loaded from file: /projects/.config/esa_maap/ticket


In [15]:
USE_BIOMASS_BBOX = False   # set True to restrict BIOMASS to AOI, False for more items

This cell queries the ESA MAAP STAC catalog for BIOMASS V5.01 items. It opens the STAC endpoint, builds a `search_kwargs` dictionary with the BIOMASS collection, the selected year/time window (`BIOMASS_DT`), and a result cap (`BIOMASS_LIMIT = 25`). If `USE_BIOMASS_BBOX` is enabled, it also adds your AOI bounding box to restrict results spatially; otherwise it searches more broadly. Finally, it executes the STAC search, collects the returned items into `biomass_items`, and prints how many BIOMASS items were found and whether the bbox filter was applied.


In [29]:
STAC_URL = "https://catalog.maap.eo.esa.int/catalogue/"
BIOMASS_COLLECTION = "CCIBiomassV5.01"

BIOMASS_LIMIT = 25  

api = Client.open(STAC_URL)

search_kwargs = dict(
    collections=[BIOMASS_COLLECTION],
    datetime=BIOMASS_DT,
    limit=BIOMASS_LIMIT,
)

if USE_BIOMASS_BBOX:
    search_kwargs["bbox"] = list(BBOX)

search = api.search(**search_kwargs)

biomass_items = list(search.get_items())
print("BIOMASS items found:", len(biomass_items))

BIOMASS items found: 619


This cell converts the BIOMASS STAC search results (`biomass_items`) into a GeoJSON `FeatureCollection` for mapping. It defines `biomass_item_to_feature()` to package each STAC item’s footprint geometry (`item.geometry`) and key metadata fields (item id and temporal fields like `start_datetime`/`end_datetime`) into a GeoJSON Feature. It then applies this conversion to every BIOMASS item, stores the results in `biomass_fc`, and prints how many BIOMASS footprint features are available to plot on the interactive map (here, 619).


In [30]:
# Build BIOMASS FeatureCollection for mapping
def biomass_item_to_feature(item):
    return {
        "type": "Feature",
        "geometry": item.geometry,
        "properties": {
            "id": item.id,
            "start_datetime": item.properties.get("start_datetime"),
            "end_datetime": item.properties.get("end_datetime"),
            "datetime": item.properties.get("datetime"),
        },
    }

biomass_features = [biomass_item_to_feature(it) for it in biomass_items]
biomass_fc = {"type": "FeatureCollection", "features": biomass_features}

print("BIOMASS footprints in FeatureCollection:", len(biomass_fc["features"]))


BIOMASS footprints in FeatureCollection: 619


## Interactive map: NISAR and BIOMASS footprint layers

This section creates a single interactive Leaflet/Folium map centered on the midpoint of the AOI `BBOX`. It then overlays two GeoJSON layers: one showing the footprint polygons for all discovered NISAR granules (`nisar_fc`) with tooltips for the granule title and start/end times, and another showing the footprint polygons for BIOMASS items (`biomass_fc`) with tooltips for tile ID and temporal coverage. Finally, it adds a `LayerControl` so you can toggle the NISAR and BIOMASS layers on/off to visually compare their spatial overlap.


In [26]:
# Create base map centered on bbox
center_lat = (BBOX[1] + BBOX[3]) / 2
center_lon = (BBOX[0] + BBOX[2]) / 2

m = Map(tiles="OpenStreetMap", location=(center_lat, center_lon), zoom_start=7)

# Add NISAR footprints
GeoJson(
    data=nisar_fc,
    name=f"NISAR ({len(nisar_fc['features'])} granules)",
    tooltip=["title", "t0", "t1"],
).add_to(m)

# Add BIOMASS footprints
GeoJson(
    data=biomass_fc,
    name=f"BIOMASS {BIOMASS_YEAR} ({len(biomass_fc['features'])} items)",
    tooltip=["id", "start_datetime", "end_datetime"],
).add_to(m)

LayerControl(collapsed=False).add_to(m)

m


## Overlap of BIOMASS tiles intersecting with NISAR granule

This cell summarizes the overlap results by counting how many overlap pairs are associated with each NISAR granule index `(nisar_i)` in `overlap_fc`. It prints the total number of NISAR granules and BIOMASS items being compared, the total number of overlap pairs found, and then lists each NISAR granule with the number of BIOMASS items that overlap it, along with the granule’s title. This helps explain why the overlap count can be large: it is counting many BIOMASS-to-one-NISAR pairings, not unique NISAR granules.

In [23]:
counts = Counter([f["properties"]["nisar_i"] for f in overlap_fc["features"]])
print("NISAR granules:", len(nisar_fc["features"]))
print("BIOMASS items:", len(biomass_fc["features"]))
print("Total overlaps (pairs):", len(overlap_fc["features"]))

for i in range(len(nisar_fc["features"])):
    title = nisar_fc["features"][i]["properties"].get("title")
    print(i, counts.get(i, 0), "-", title)


NISAR granules: 6
BIOMASS items: 619
Total overlaps (pairs): 134
0 22 - NISAR_L2_PR_GCOV_003_005_D_077_4005_DHDH_A_20251017T132451_20251017T132526_X05007_N_F_J_001
1 20 - NISAR_L2_PR_GCOV_003_064_D_130_7700_SHNA_A_20251021T160803_20251021T160836_X05007_N_P_J_001
2 20 - NISAR_L2_PR_GCOV_004_064_D_130_7700_SHNA_A_20251102T160804_20251102T160837_X05007_N_P_J_001
3 28 - NISAR_L2_PR_GCOV_004_076_A_022_2005_QPDH_A_20251103T110514_20251103T110549_X05007_N_F_J_002
4 22 - NISAR_L2_PR_GCOV_005_172_A_008_2005_DHDH_A_20251122T024618_20251122T024652_X05007_N_F_J_001
5 22 - NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05007_N_F_J_001


This cell computes and visualizes a *bounding-box* overlap layer between the two datasets. First, `geom_bbox_polygon()` converts each feature’s footprint geometry into its rectangular bounding box using `.bounds`, so overlap checks are fast and consistent. It then loops over every NISAR feature and every BIOMASS feature, intersects their bounding boxes (`nb.intersection(bb)`), and whenever the intersection is non-empty it records an “overlap pair” as a new GeoJSON Feature with metadata from both items (NISAR title/time and BIOMASS id/time). All overlap features are collected into `overlap_fc`, the total number of bbox-overlap pairs is printed, and the overlap intersections are added as a third toggleable Folium layer (“Overlap (bbox ∩ bbox)”) so you can click/hover to inspect which NISAR and BIOMASS items overlap in space.


In [22]:
def geom_bbox_polygon(feature):
    g = shape(feature["geometry"])
    minx, miny, maxx, maxy = g.bounds
    return box(minx, miny, maxx, maxy)

overlap_features = []

nisar_feats = nisar_fc["features"]
biomass_feats = biomass_fc["features"]

for i, nf in enumerate(nisar_feats):
    nb = geom_bbox_polygon(nf)

    for j, bf in enumerate(biomass_feats):
        bb = geom_bbox_polygon(bf)

        inter = nb.intersection(bb)
        if not inter.is_empty:
            overlap_features.append({
                "type": "Feature",
                "geometry": mapping(inter),
                "properties": {
                    "nisar_i": i,
                    "biomass_j": j,
                    "nisar_title": nf["properties"].get("title"),
                    "nisar_t0": nf["properties"].get("t0"),
                    "nisar_t1": nf["properties"].get("t1"),
                    "biomass_id": bf["properties"].get("id"),
                    "biomass_start": bf["properties"].get("start_datetime"),
                    "biomass_end": bf["properties"].get("end_datetime"),
                }
            })

overlap_fc = {"type": "FeatureCollection", "features": overlap_features}
print("BBox overlaps found:", len(overlap_fc["features"]))


GeoJson(
    data=overlap_fc,
    name="Overlap (bbox ∩ bbox)",
    tooltip=[
        "nisar_title", "nisar_t0", "nisar_t1",
        "biomass_id", "biomass_start", "biomass_end"
    ],
).add_to(m)

LayerControl(collapsed=False).add_to(m)
m


BBox overlaps found: 134
